In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [8]:
data = np.load("dayabay_analysis_dataset_npz_1-0-0/npz/dayabay_dataset/dayabay_background_rates.npz", allow_pickle=True)
arr = data['6AD']
print(arr)

[('accidentals_rate', 8.39901  , 8.18397  , 6.65972  , 0, 2.62981  , 2.60203  , 2.54534  , 0)
 ('accidentals_uncertainty', 0.0257708, 0.0252693, 0.0217033, 0, 0.0128695, 0.0128775, 0.0127418, 0)
 ('lithium_helium_rate', 2.92     , 2.92     , 2.45     , 0, 0.26     , 0.26     , 0.26     , 0)
 ('lithium_helium_uncertainty', 0.78     , 0.78     , 0.57     , 0, 0.04     , 0.04     , 0.04     , 0)
 ('fast_neutrons_rate', 0.791    , 0.791    , 0.503    , 0, 0.04     , 0.04     , 0.04     , 0)
 ('fast_neutrons_uncertainty', 0.1582   , 0.1582   , 0.1006   , 0, 0.008    , 0.008    , 0.008    , 0)
 ('amc_rate', 0.277217 , 0.268434 , 0.296314 , 0, 0.230175 , 0.22248  , 0.219081 , 0)
 ('amc_uncertainty', 0.117613 , 0.113887 , 0.125715 , 0, 0.0976551, 0.0943905, 0.0929483, 0)
 ('alpha_neutron_rate', 0.09     , 0.07     , 0.05     , 0, 0.05     , 0.04     , 0.04     , 0)
 ('alpha_neutron_uncertainty', 0.045    , 0.035    , 0.025    , 0, 0.025    , 0.02     , 0.02     , 0)]


In [14]:
import numpy as np

base_path = "dayabay_full_dataset_npz_1-0-0/npz/dayabay_dataset"
detectors = ["AD11", "AD12", "AD21", "AD22", "AD31", "AD32", "AD33", "AD34"]

events = {}
for detector in detectors:
    data = np.load(f"{base_path}/dayabay_events_{detector}.npz")
    events[detector] = data["events"]
    data.close()

print(events)

{'AD11': array([(1324738631,    0, 6, 4.39337, 7.8336 , 21.875,   517.856, -1408.62 ,  -33.650303,   354.15598, -1100.27 ,   -93.051796),
       (1324738749,    0, 6, 3.81   , 7.86787, 18.775,  -939.942,   484.862,  234.307   ,  -955.195  ,   579.852,   187.133   ),
       (1324738785,    0, 6, 4.72838, 8.4689 , 14.425, -1235.02 ,   770.802, -929.329   , -1275.49   ,   382.811, -1020.56    ),
       ...,
       (1482277170, 1823, 8, 4.9176 , 7.71446, 36.075,  -964.552, -1149.32 ,   77.49731 , -1226.8601 ,  -950.471,   124.394   ),
       (1482277187, 1823, 8, 5.79654, 8.25297, 13.863,  -350.537,  1062.8  ,  491.438   ,  -392.486  ,  1036.79 ,   528.079   ),
       (1482277248, 1823, 8, 1.43829, 8.04153, 64.513,  1136.47 ,  -755.689, -140.33601 ,   970.235  , -1044.14 ,  -362.82    )],
      shape=(794335,), dtype=[('time_prompt_s', '<u4'), ('day', '<u2'), ('n_det', 'u1'), ('energy_prompt_MeV', '<f4'), ('energy_delayed_MeV', '<f4'), ('delta_t_us', '<f4'), ('vertex_prompt_x_mm', '<f4'), 

In [45]:
def count_events_by_period(events, n_days=7):
    n_periods = max(
        (int(data["day"].max() // n_days) + 1 for data in events.values() if len(data)),
        default=0,
    )

    return {
        detector: np.bincount(
            (data["day"] // n_days).astype(int),
            minlength=n_periods,
        )
        for detector, data in events.items()
    }


weekly_counts = count_events_by_period(events)
monthly_counts = count_events_by_period(events, n_days=30)

In [51]:
import pandas as pd


def analyze_correlations(events, n_days=7):
    detectors = list(events)

    max_period = max(
        int(data["day"].max() // n_days)
        for data in events.values()
    )

    counts = pd.DataFrame(
        0,
        index=pd.RangeIndex(max_period + 1, name="period"),
        columns=detectors,
        dtype=int,
    )

    for detector, data in events.items():
        periods, numbers = np.unique(
            (data["day"] // n_days).astype(int),
            return_counts=True,
        )
        counts.loc[periods, detector] = numbers

    pearson = pd.DataFrame(
        np.corrcoef(counts.to_numpy(), rowvar=False),
        index=detectors,
        columns=detectors,
    )

    return pearson


pearson = analyze_correlations(events, n_days=7)

print(pearson.round(3))

       AD11   AD12   AD21   AD22   AD31   AD32   AD33   AD34
AD11  1.000  0.330  0.321  0.111  0.329  0.330  0.339  0.127
AD12  0.330  1.000  0.830  0.643  0.899  0.898  0.895  0.684
AD21  0.321  0.830  1.000  0.773  0.976  0.975  0.974  0.749
AD22  0.111  0.643  0.773  1.000  0.753  0.756  0.759  0.984
AD31  0.329  0.899  0.976  0.753  1.000  0.979  0.979  0.747
AD32  0.330  0.898  0.975  0.756  0.979  1.000  0.981  0.753
AD33  0.339  0.895  0.974  0.759  0.979  0.981  1.000  0.755
AD34  0.127  0.684  0.749  0.984  0.747  0.753  0.755  1.000


In [43]:
seed, N, p, M = input().split()
N, M, seed = int(N), int(M), int(seed)
p = float(p)


diff = [N] * (M + 1)
total = [N] * (M + 1)

np.random.seed(seed)
for i in range(M):
    res = np.random.binomial(1, p, total[i])
    total[i + 1] = total[i] - np.sum(res)
    # diff[i + 1] = diff[i] - np.sum(1 - res)
    diff[i + 1] = diff[i] + np.sum(res)

for i in range(1, len(diff)):
    print((diff[i] - diff[i - 1])/diff[i])

print(total)

0.3106642990283124
0.14622157352041057
0.07311559543343923
0.03863193931664093
0.021856712546885645
0.011063132092011494
0.006507592190889371
[62500, np.int64(34333), np.int64(18805), np.int64(10428), np.int64(5824), np.int64(3161), np.int64(1798), np.int64(991)]


In [31]:
import numpy as np

seed, N, p, M = input().split()
seed, N, M = int(seed), int(N), int(M)
p = float(p)

np.random.seed(seed)

# 1 represents heads; 0 represents tails
flips = np.random.binomial(1, p, size=(M, N))

# A coin survives only if it lands tails in every iteration so far
survived = np.cumprod(flips == 0, axis=0)

diff = np.concatenate(([N], survived.sum(axis=1)))

result = diff[1:] / diff[:-1]

print(*(f"{x:.3f}" for x in result))

0.506 0.514 0.495 0.493 0.520 0.515 0.535 0.560 0.549


In [44]:
import numpy as np

seed, N, p, M = input().split()
seed = int(seed)
N = int(N)
p = float(p)
M = int(M)

np.random.seed(seed)

diff = []

for _ in range(M):
    tosses = np.random.binomial(1, p, size=N)
    heads = np.sum(tosses)

    diff.append((N - heads) / N)
    N = heads

print(*[f"{x:.3f}" for x in diff])

0.549 0.548 0.551 0.554 0.556 0.564 0.585
